# How to Use This Notebook

This notebook is designed to be run sequentially from top to bottom without manual intervention. The cells are grouped into numbered steps. Please execute each step in order to run the verification.

- **Step 1: Environment Setup:** Prepares the Kaggle environment, installs dependencies, and verifies TPU access.
- **Step 2: Apply Compatibility Fix:** Downgrades NumPy to prevent known version conflicts.
- **Step 3: Configure Checkpoint Path:** Finds the Llama 3.1 checkpoint dataset and sets the required environment variable.
- **Step 4: Generate Config and Run Verification:** Uses the verified `load_parameters_path` key to generate the complete YAML file and runs the final 1-step training verification. Success is indicated by a "Verification run completed successfully" message.


# Step 1: Environment Setup

This step prepares the Kaggle environment by:
1.  **Verifying JAX and TPU Access:** Ensures the notebook can see the 8 TPU devices.
2.  **Cloning MaxText:** Clones the `google/maxtext` repository, which contains the training scripts.
3.  **Installing Dependencies:** Installs all Python packages required by MaxText from `requirements.txt`.


In [1]:
import os, sys, platform, subprocess, shutil

print("Verifying JAX and TPU environment...")
try:
    import jax
    import jax.numpy as jnp
    device_count = jax.device_count()
    print(f"✅ JAX version: {jax.__version__}")
    print(f"✅ Detected {device_count} TPU devices.")
    if device_count != 8:
        print("⚠️ WARNING: Expected 8 TPU devices, but found a different number.")
except Exception as e:
    print(f"❌ ERROR: JAX/TPU verification failed: {e}")
    raise

# Nuke and Pave: Delete the repo if it exists to ensure a clean clone.
print("\nEnsuring a clean state for MaxText repository...")
if os.path.exists('maxtext'):
    shutil.rmtree('maxtext')
    print("✅ Removed existing 'maxtext' directory.")

# Perform a fresh, full clone.
print("Cloning fresh copy of MaxText repository...")
subprocess.run(["git", "clone", "https://github.com/google/maxtext.git"], check=True)
print("✅ MaxText repository cloned.")

# This is the parent commit of a55e18a, which introduced a breaking change
# by importing `colocated_python`, a feature not present in JAX 0.4.34.
# Using this commit is a deterministic, evidence-based fix.
stable_commit_hash = "c581c815858f09070057088272379d473489000a"
print(f"Checking out stable MaxText commit: {stable_commit_hash[:10]}...")
subprocess.run(["git", "checkout", stable_commit_hash], cwd="maxtext", check=True)
print("✅ Git checkout successful.")


print("\nInstalling dependencies...")
subprocess.run(["apt-get", "update"], check=True, capture_output=True)
subprocess.run(["apt-get", "install", "-y", "pkg-config"], check=True, capture_output=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "pip"], check=True, capture_output=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "maxtext/requirements.txt"], check=True, capture_output=True)
print("✅ Dependencies installed.")


Verifying JAX and TPU environment...


E0000 00:00:1758139467.872104    9381 common_lib.cc:612] Could not set metric server port: INVALID_ARGUMENT: Could not find SliceBuilder port 8471 in any of the 0 ports provided in `tpu_process_addresses`="local"
=== Source Location Trace: ===
learning/45eac/tfrc/runtime/common_lib.cc:230


✅ JAX version: 0.4.34
✅ Detected 8 TPU devices.

Ensuring a clean state for MaxText repository...
✅ Removed existing 'maxtext' directory.
Cloning fresh copy of MaxText repository...


Cloning into 'maxtext'...


✅ MaxText repository cloned.
Checking out stable MaxText commit: c581c81585...


fatal: reference is not a tree: c581c815858f09070057088272379d473489000a


CalledProcessError: Command '['git', 'checkout', 'c581c815858f09070057088272379d473489000a']' returned non-zero exit status 128.

# Step 2: Apply Compatibility Fix

This step downgrades NumPy to version 1.26.4. This is a mandatory step to prevent known compatibility issues between the pre-installed TensorFlow and NumPy 2.x in the Kaggle TPU environment.


In [ ]:
import sys
import subprocess

print("Applying NumPy compatibility fix...")
subprocess.run([sys.executable, "-m", "pip", "install", "numpy<2"], check=True, capture_output=True)

print("Verifying NumPy version...")
# We run this in a subprocess to ensure we get the version from the updated environment
result = subprocess.run([sys.executable, "-c", "import numpy as np; print(np.__version__)"], check=True, capture_output=True, text=True)
numpy_version = result.stdout.strip()
print(f"✅ NumPy version is now: {numpy_version}")

if numpy_version != "1.26.4":
    print("⚠️ WARNING: Expected NumPy 1.26.4, but a different version is installed. This may cause issues.")
else:
    print("✅ NumPy version successfully downgraded to 1.26.4.")

print("\nNOTE: A kernel restart may be required for the version change to fully propagate in all contexts.")


# Step 3: Configure Checkpoint Path

This step dynamically locates the pre-converted Llama 3.1 MaxText checkpoint within the attached Kaggle Datasets and sets the `MAXTEXT_CHECKPOINT_DIR` environment variable. This makes the checkpoint path available for the final verification step.


In [ ]:
import os
from pathlib import Path

dataset_path = Path("/kaggle/input/llama-3-1-8b-maxtext-checkpoint")

print(f"Inspecting dataset directory: {dataset_path}")
if not dataset_path.exists():
    raise FileNotFoundError(f"Dataset directory not found: {dataset_path}")

required_files = ["_CHECKPOINT_METADATA", "items"]
if all((dataset_path / f).exists() for f in required_files):
    checkpoint_dir = dataset_path
    print(f"✅ Checkpoint found in root directory: {checkpoint_dir}")
else:
    raise FileNotFoundError(f"Could not find required checkpoint files in {dataset_path}")

os.environ["MAXTEXT_CHECKPOINT_DIR"] = str(checkpoint_dir)
print(f"✅ Environment variable set: MAXTEXT_CHECKPOINT_DIR={os.environ['MAXTEXT_CHECKPOINT_DIR']}")


# Step 4: Generate Config and Run Verification

This is the final, fully automated step. It performs the following actions:

1.  **Sets `PYTHONPATH`:** Ensures the MaxText library can be correctly imported.
2.  **Generates YAML:** Creates the `verification_minimal.yml` file using the verified `load_parameters_path` key and the checkpoint path from the previous step.
3.  **Runs Verification:** Executes the MaxText training script as a module (`MaxText.train`) for a single step. 

A successful run will print "✅ Verification run completed successfully." and is the evidence that the entire environment is correctly configured.


In [ ]:
import os
import sys
import subprocess
from pathlib import Path

# Set environment variable to resolve TensorFlow protobuf conflict
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"

# 1. Add maxtext to python path for imports
maxtext_src_path = str(Path.cwd() / "maxtext" / "src")
if maxtext_src_path not in sys.path:
    sys.path.insert(0, maxtext_src_path)
os.environ['PYTHONPATH'] = f"{maxtext_src_path}:{os.environ.get('PYTHONPATH', '')}"
print(f"✅ PYTHONPATH set to include: {maxtext_src_path}")

# 2. Get checkpoint path and define the verified key
checkpoint_path = os.environ.get("MAXTEXT_CHECKPOINT_DIR")
if not checkpoint_path:
    raise ValueError("MAXTEXT_CHECKPOINT_DIR not set. Run the previous step first.")
    
verified_checkpoint_key = "load_parameters_path" # Verified from source code
print(f"✅ Using verified key '{verified_checkpoint_key}' for checkpoint loading.")

# 3. Generate the complete and correct YAML configuration
config_text = f"""
# Auto-generated configuration for verification run
run_name: "verification_run_1step"
base_output_directory: "/kaggle/working/maxtext_runs"
steps: 1
per_device_batch_size: 1
model_name: "llama3.1-8b"
ici_parallelism: -1
mesh_axis_names: ['data', 'fsdp', 'tensor']
{verified_checkpoint_key}: "{checkpoint_path}"
"""

config_path = Path("/kaggle/working/verification_minimal.yml")
config_path.write_text(config_text)
print(f"✅ Wrote config to: {config_path}")
print("--- Config Contents ---")
print(config_text)
print("-----------------------")

# 4. Run the verification script as a module
print("\n🚀 Running 1-step verification...")

cmd = [
    sys.executable, "-m", "MaxText.train", str(config_path)
]

try:
    # Using Popen to stream output in real-time
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=os.environ)
    while True:
        output = process.stdout.readline()
        if output == '' and process.poll() is not None:
            break
        if output:
            print(output.strip())
    
    rc = process.poll()
    if rc == 0:
        print("\n✅ Verification run completed successfully.")
    else:
        print(f"\n❌ Verification run failed with return code {rc}")

except FileNotFoundError:
    print("❌ ERROR: Could not find the training module. Is MaxText cloned correctly?")
except Exception as e:
    print(f"❌ An unexpected error occurred: {e}")

